In [1]:
import os
from google.colab import drive

# --- 1. SETUP AND INSTALLATION ---
print("Installing required packages...")
# Using -qq to make the output less noisy
# Added 'datasets' for loading data and 'jiwer' for WER/CER calculation
!pip install -qq flask pyngrok transformers torch torchaudio safetensors datasets jiwer torchcodec

print("\nMounting Google Drive...")
drive.mount('/content/drive')

# --- 2. CREATE DIRECTORIES FOR THE FLASK APP ---
print("Creating app directories...")
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

# --- 3. WRITE THE MODEL LOGIC (model.py) ---
# This file will contain the classes and functions to load and run all three of your models.
print("Creating model.py...")
model_py_content = """
import torch
import torch.nn as nn
import torchaudio
from transformers import Wav2Vec2Processor, pipeline
import os
from safetensors.torch import load_file
import warnings
from datasets import load_dataset, Audio
import jiwer

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore")

# ==============================================================================
# 1. RE-DEFINE THE CUSTOM CNN-RNN-CTC MODEL ARCHITECTURE
# This must match the architecture used during training.
# ==============================================================================
class ASRModel(nn.Module):
    def __init__(self, n_mels, rnn_dim, vocab_size, n_cnn_layers, n_rnn_layers, dropout):
        super().__init__()
        cnn_channels = 32
        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1, stride=1), nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1, stride=1), nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.MaxPool2d(kernel_size=(2, 1)),
        )
        cnn_output_dim = (n_mels // 2) * cnn_channels
        self.rnn = nn.GRU(
            input_size=cnn_output_dim, hidden_size=rnn_dim,
            num_layers=n_rnn_layers, bidirectional=True,
            batch_first=True, dropout=dropout if n_rnn_layers > 1 else 0
        )
        self.classifier = nn.Linear(rnn_dim * 2, vocab_size)

    def forward(self, input_values, **kwargs):
        x = input_values.unsqueeze(1)
        x = self.cnn(x)
        batch_size, channels, freq_dim, seq_len = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, seq_len, -1)
        x, _ = self.rnn(x)
        logits = self.classifier(x)
        from transformers.modeling_outputs import CausalLMOutput
        return CausalLMOutput(logits=logits)


# ==============================================================================
# 2. MAIN INFERENCE CLASS TO WRAP ALL MODELS
# This class will load all models and provide a single predict method.
# ==============================================================================
class MultiModelInference:
    def __init__(self, conformer_path, wav2vec2_path, custom_model_path):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Initializing system on device: {self.device}")

        # --- Load Conformer and Wav2Vec2 via Transformers Pipeline ---
        print("Loading Conformer model...")
        self.conformer_pipe = pipeline("automatic-speech-recognition", model=conformer_path, device=0 if self.device.type == 'cuda' else -1)
        print("Loading Wav2Vec2 model...")
        self.wav2vec2_pipe = pipeline("automatic-speech-recognition", model=wav2vec2_path, device=0 if self.device.type == 'cuda' else -1)

        # --- Load Custom CNN-RNN-CTC Model ---
        print("Loading custom CNN-RNN-CTC model...")
        self.custom_model_dir = custom_model_path
        self.custom_processor = Wav2Vec2Processor.from_pretrained(self.custom_model_dir)
        self.custom_tokenizer = self.custom_processor.tokenizer

        # Define model parameters (must match training)
        self.custom_model = ASRModel(
            n_mels=80, rnn_dim=512, vocab_size=29,
            n_cnn_layers=3, n_rnn_layers=3, dropout=0.1
        )
        weights_path = os.path.join(self.custom_model_dir, "model.safetensors")
        state_dict = load_file(weights_path, device="cpu")
        self.custom_model.load_state_dict(state_dict)
        self.custom_model.to(self.device).eval()
        print("✅ All models loaded successfully.")

        # --- Audio Processing Transforms ---
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
        ).to(self.device)
        self.resampler = None

    @torch.no_grad()
    def _predict_custom(self, waveform):
        # Preprocess audio for the custom model
        mel_spec = self.mel_transform(waveform.to(self.device))
        log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5))

        # Run inference
        output = self.custom_model(log_mel_spec)
        logits = output.logits
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = self.custom_tokenizer.batch_decode(predicted_ids)[0]
        return transcription

    def predict(self, audio_path):
        # Load waveform for all models
        waveform_tensor, sr = torchaudio.load(audio_path)
        if sr != 16000:
            if self.resampler is None:
                 self.resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            waveform_tensor = self.resampler(waveform_tensor)
        if waveform_tensor.shape[0] > 1: # Convert to mono
            waveform_tensor = torch.mean(waveform_tensor, dim=0, keepdim=True)

        # For pipeline, convert to numpy array
        waveform_np = waveform_tensor.squeeze().numpy()

        # --- Conformer Prediction ---
        transcription_conformer = self.conformer_pipe(waveform_np.copy())["text"]

        # --- Wav2Vec2 Prediction ---
        raw_wav2vec2 = self.wav2vec2_pipe(waveform_np.copy())["text"]
        # Clean up [PAD] tokens and extra whitespace
        clean_wav2vec2 = raw_wav2vec2.replace('[PAD]', '')
        transcription_wav2vec2 = " ".join(clean_wav2vec2.split())

        # --- Custom Model Prediction ---
        transcription_custom = self._predict_custom(waveform_tensor)

        return {
            "Conformer": transcription_conformer,
            "Wav2Vec2": transcription_wav2vec2,
            "CNN-RNN-CTC": transcription_custom,
        }

    def evaluate(self, dataset_name):
        print(f"Loading dataset {dataset_name} for evaluation...")
        # Using a small subset for quick demonstration. Remove .select() for full evaluation.
        # FIX: The dataset does not have a language_code column. Evaluating on a subset of the whole test set.
        full_test_dataset = load_dataset(dataset_name, split="test")
        test_dataset = full_test_dataset.select(range(min(20, len(full_test_dataset)))) # Use a subset

        # The 'audio' feature needs to be resampled to 16kHz for model consistency
        test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

        predictions = {"Conformer": [], "Wav2Vec2": [], "CNN-RNN-CTC": []}
        references = []

        print("Starting evaluation...")
        for i, item in enumerate(test_dataset):
            print(f"Processing item {i+1}/{len(test_dataset)}...")
            audio_input = item["audio"]["array"]
            # CORRECTED: Use 'transcription' as the key for the reference text, based on the dataset card
            reference_text = item["transcription"]
            references.append(reference_text)

            # --- Conformer Prediction ---
            pred_conformer = self.conformer_pipe(audio_input.copy())["text"]
            predictions["Conformer"].append(pred_conformer)

            # --- Wav2Vec2 Prediction ---
            raw_wav2vec2 = self.wav2vec2_pipe(audio_input.copy())["text"]
            clean_wav2vec2 = raw_wav2vec2.replace('[PAD]', '')
            pred_wav2vec2 = " ".join(clean_wav2vec2.split())
            predictions["Wav2Vec2"].append(pred_wav2vec2)

            # --- Custom Model Prediction ---
            waveform = torch.tensor(audio_input).unsqueeze(0)
            pred_custom = self._predict_custom(waveform)
            predictions["CNN-RNN-CTC"].append(pred_custom)

        results = {}
        print("\\nCalculating WER/CER scores...")
        for model_name, preds in predictions.items():
            wer = jiwer.wer(references, preds)
            cer = jiwer.cer(references, preds)
            results[model_name] = {"WER": f"{wer:.4f}", "CER": f"{cer:.4f}"}
            print(f"Results for {model_name}: WER={wer:.4f}, CER={cer:.4f}")

        return results

"""
with open("model.py", "w") as f:
    f.write(model_py_content)


# --- 4. WRITE THE WEB INTERFACE (templates/index.html) ---
print("Creating templates/index.html...")
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Multi-Model ASR Inference</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        @import url('https://rsms.me/inter/inter.css');
        body { font-family: 'Inter', sans-serif; }
        .result-card, .eval-card {
            transition: all 0.3s ease-in-out;
        }
    </style>
</head>
<body class="bg-gray-50 flex items-center justify-center min-h-screen py-12">
    <div class="bg-white p-8 rounded-xl shadow-lg w-full max-w-3xl m-4">
        <h1 class="text-3xl font-bold text-gray-800 mb-2">Multi-Model ASR Inference</h1>
        <p class="text-gray-600 mb-6">Upload an audio file for transcription or evaluate models on the NCHLT test set.</p>

        <form id="upload-form" class="space-y-4">
            <div>
                <div class="mt-1 flex justify-center px-6 pt-5 pb-6 border-2 border-gray-300 border-dashed rounded-md">
                    <div class="space-y-1 text-center">
                        <svg class="mx-auto h-12 w-12 text-gray-400" stroke="currentColor" fill="none" viewBox="0 0 48 48" aria-hidden="true"><path d="M28 8H12a4 4 0 00-4 4v20m32-12v8m0 0v8a4 4 0 01-4 4H12a4 4 0 01-4-4v-4m32-4l-3.172-3.172a4 4 0 00-5.656 0L28 28M8 32l9.172-9.172a4 4 0 015.656 0L28 28m0 0l4 4m4-24h8m-4-4v8" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" /></svg>
                        <div class="flex text-sm text-gray-600">
                            <label for="audio-file-input" class="relative cursor-pointer bg-white rounded-md font-medium text-indigo-600 hover:text-indigo-500">
                                <span>Upload a file</span>
                                <input id="audio-file-input" name="audio" type="file" class="sr-only" accept="audio/wav, audio/mpeg">
                            </label>
                            <p class="pl-1">or drag and drop</p>
                        </div>
                        <p id="file-name" class="text-xs text-gray-500">WAV, MP3 up to 10MB</p>
                    </div>
                </div>
            </div>
            <div>
                <button type="submit" id="submit-btn" class="w-full flex justify-center py-3 px-4 border border-transparent rounded-md shadow-sm text-sm font-medium text-white bg-indigo-600 hover:bg-indigo-700 focus:outline-none focus:ring-2 focus:ring-offset-2 focus:ring-indigo-500">
                    Transcribe
                </button>
            </div>
             <div>
                <button type="button" id="evaluate-btn" class="w-full flex justify-center py-3 px-4 border border-transparent rounded-md shadow-sm text-sm font-medium text-white bg-emerald-600 hover:bg-emerald-700 focus:outline-none focus:ring-2 focus:ring-offset-2 focus:ring-emerald-500">
                    Evaluate Models on NCHLT Test Set
                </button>
            </div>
        </form>

        <div id="results" class="mt-8 hidden">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Transcription Results</h2>
            <div id="loader" class="text-center py-4 hidden">
                <div class="animate-spin rounded-full h-8 w-8 border-b-2 border-indigo-500 mx-auto"></div>
                <p class="mt-2 text-gray-600">Processing audio with all models, please wait...</p>
            </div>
            <div id="result-content" class="space-y-4"></div>
        </div>

        <div id="evaluation-results" class="mt-8 hidden">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Evaluation Metrics (WER / CER)</h2>
            <div id="evaluation-loader" class="text-center py-4 hidden">
                <div class="animate-spin rounded-full h-8 w-8 border-b-2 border-emerald-500 mx-auto"></div>
                <p class="mt-2 text-gray-600">Evaluating on a test subset, this may take a few minutes...</p>
            </div>
            <div id="evaluation-content"></div>
        </div>
    </div>

    <script>
        const form = document.getElementById('upload-form');
        const fileInput = document.getElementById('audio-file-input');
        const fileNameDisplay = document.getElementById('file-name');
        const submitBtn = document.getElementById('submit-btn');

        const resultsDiv = document.getElementById('results');
        const loader = document.getElementById('loader');
        const resultContent = document.getElementById('result-content');

        const evaluateBtn = document.getElementById('evaluate-btn');
        const evalResultsDiv = document.getElementById('evaluation-results');
        const evalLoader = document.getElementById('evaluation-loader');
        const evalContent = document.getElementById('evaluation-content');

        fileInput.addEventListener('change', () => {
            fileNameDisplay.textContent = fileInput.files.length > 0 ? fileInput.files[0].name : 'WAV, MP3 up to 10MB';
        });

        form.addEventListener('submit', async (e) => {
            e.preventDefault();
            if (fileInput.files.length === 0) {
                alert('Please select an audio file first.');
                return;
            }

            const formData = new FormData();
            formData.append('audio', fileInput.files[0]);

            resultsDiv.classList.remove('hidden');
            evalResultsDiv.classList.add('hidden'); // Hide eval results
            loader.classList.remove('hidden');
            resultContent.innerHTML = '';
            submitBtn.disabled = true;
            evaluateBtn.disabled = true;
            submitBtn.classList.add('opacity-50', 'cursor-not-allowed');
            evaluateBtn.classList.add('opacity-50', 'cursor-not-allowed');

            try {
                const response = await fetch('/predict', { method: 'POST', body: formData });

                if (!response.ok) {
                    const error = await response.json();
                    throw new Error(error.error || 'An unknown error occurred');
                }

                const data = await response.json();
                let html = '';
                const colors = { 'Conformer': 'indigo', 'Wav2Vec2': 'blue', 'CNN-RNN-CTC': 'emerald' };

                for (const modelName in data) {
                    html += `<div class="result-card p-4 rounded-lg bg-${colors[modelName]}-50 border-l-4 border-${colors[modelName]}-500">
                        <h3 class="text-lg font-semibold text-${colors[modelName]}-800">${modelName}</h3>
                        <p class="mt-1 text-md text-gray-900">${data[modelName] || 'No transcription.'}</p>
                    </div>`;
                }
                resultContent.innerHTML = html;

            } catch (error) {
                resultContent.innerHTML = `<div class="p-4 rounded-lg bg-red-50 border-l-4 border-red-500"><h3 class="text-lg font-semibold text-red-800">Error</h3><p class="mt-1 text-md text-red-900">${error.message}</p></div>`;
            } finally {
                loader.classList.add('hidden');
                submitBtn.disabled = false;
                evaluateBtn.disabled = false;
                submitBtn.classList.remove('opacity-50', 'cursor-not-allowed');
                evaluateBtn.classList.remove('opacity-50', 'cursor-not-allowed');
            }
        });

        evaluateBtn.addEventListener('click', async () => {
            evalResultsDiv.classList.remove('hidden');
            resultsDiv.classList.add('hidden'); // Hide transcription results
            evalLoader.classList.remove('hidden');
            evalContent.innerHTML = '';
            evaluateBtn.disabled = true;
            submitBtn.disabled = true;
            evaluateBtn.classList.add('opacity-50', 'cursor-not-allowed');
            submitBtn.classList.add('opacity-50', 'cursor-not-allowed');

            try {
                const response = await fetch('/evaluate', { method: 'POST' });

                if (!response.ok) {
                    const error = await response.json();
                    throw new Error(error.error || 'An unknown error occurred during evaluation');
                }

                const data = await response.json();
                let html = '<div class="grid grid-cols-1 md:grid-cols-3 gap-4">';
                const colors = { 'Conformer': 'indigo', 'Wav2Vec2': 'blue', 'CNN-RNN-CTC': 'emerald' };

                for (const modelName in data) {
                    html += `<div class="eval-card p-4 rounded-lg bg-${colors[modelName]}-50 border-t-4 border-${colors[modelName]}-500 text-center shadow-md">
                        <h3 class="text-lg font-bold text-${colors[modelName]}-800">${modelName}</h3>
                        <div class="mt-2"><p class="text-sm text-gray-600">Word Error Rate</p><p class="text-2xl font-semibold text-gray-900">${data[modelName].WER}</p></div>
                        <div class="mt-2"><p class="text-sm text-gray-600">Character Error Rate</p><p class="text-2xl font-semibold text-gray-900">${data[modelName].CER}</p></div>
                    </div>`;
                }
                html += '</div>';
                evalContent.innerHTML = html;

            } catch (error) {
                evalContent.innerHTML = `<div class="p-4 rounded-lg bg-red-50 border-l-4 border-red-500"><h3 class="text-lg font-semibold text-red-800">Evaluation Error</h3><p class="mt-1 text-md text-red-900">${error.message}</p></div>`;
            } finally {
                evalLoader.classList.add('hidden');
                evaluateBtn.disabled = false;
                submitBtn.disabled = false;
                evaluateBtn.classList.remove('opacity-50', 'cursor-not-allowed');
                submitBtn.classList.remove('opacity-50', 'cursor-not-allowed');
            }
        });
    </script>
</body>
</html>
"""
with open("templates/index.html", "w") as f:
    f.write(html_content)


# --- 5. WRITE THE FLASK APP (app.py) ---
print("Creating app.py...")
app_py_content = """
from flask import Flask, request, render_template, jsonify
from pyngrok import ngrok
import os
from model import MultiModelInference
import torch

# IMPORTANT: SET YOUR NGROK AUTH TOKEN HERE
# Get it from here: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "32NqxOiMohuBmUDNcFK3fAHyzvK_4Vwcpnq7PUQWR4qF3c9Xc"

if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN_HERE" or not NGROK_AUTH_TOKEN:
    print("ERROR: Ngrok authtoken is not set.")
    print("Please get your token and paste it into the NGROK_AUTH_TOKEN variable.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = Flask(__name__)

# --- DEFINE MODEL PATHS ---
# Make sure these paths are correct in your Google Drive
CONFORMER_MODEL_PATH = "/content/drive/MyDrive/wav2vecconformer-finetune-checkpoints/"
WAV2VEC2_MODEL_PATH = "/content/drive/MyDrive/wav2vec2-finetune-checkpoints22/checkpoint-20000"
CUSTOM_MODEL_PATH = "/content/drive/MyDrive/my_final_wav2vec2_model"

# --- LOAD MODELS ONCE AT STARTUP ---
print("Starting model loading process...")
inference_system = None
try:
    inference_system = MultiModelInference(
        conformer_path=CONFORMER_MODEL_PATH,
        wav2vec2_path=WAV2VEC2_MODEL_PATH,
        custom_model_path=CUSTOM_MODEL_PATH
    )
    print("\\n🚀 Model loading complete! The app is ready to use.")
except Exception as e:
    print(f"\\n❌ Error loading models: {e}")
    print("Please ensure model paths are correct and all necessary files are accessible.")

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if inference_system is None:
        return jsonify({'error': 'Models are not loaded. Check server logs for errors.'}), 500

    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file provided'}), 400

    file = request.files['audio']
    if file.filename == '':
        return jsonify({'error': 'No file selected'}), 400

    temp_path = "temp_audio_file"
    try:
        file.save(temp_path)
        result = inference_system.predict(temp_path)
        return jsonify(result)

    except Exception as e:
        print(f"Error during prediction: {e}")
        return jsonify({'error': str(e)}), 500

    finally:
        # Clean up the temporary file
        if os.path.exists(temp_path):
            os.remove(temp_path)

@app.route('/evaluate', methods=['POST'])
def evaluate():
    if inference_system is None:
        return jsonify({'error': 'Models are not loaded. Check server logs for errors.'}), 500
    try:
        # The dataset name is hardcoded for this specific request
        dataset_name = "aconeil/nchlt"
        results = inference_system.evaluate(dataset_name)
        return jsonify(results)
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
        public_url = ngrok.connect(5000)
        print(f" * ngrok tunnel is active at: {public_url}")
        app.run()
    else:
        print("Flask app did not start because NGROK_AUTH_TOKEN is not set.")

"""
with open("app.py", "w") as f:
    f.write(app_py_content)

print("\n--- ✅ Setup Complete ---")
print("You can now run the app by executing the next cell.")
print("Running the command: !python app.py")





Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.2 MB/s eta 0:00:00

Mounting Google Drive...
Mounted at /content/drive
Creating app directories...
Creating model.py...
Creating templates/index.html...
Creating app.py...

--- ✅ Setup Complete ---
You can now run the app by executing the next cell.
Running the command: !python app.py


In [ ]:
# --- 6. RUN THE FLASK APP ---
!python app.py


2025-09-10 18:14:54.972754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757528095.009170    1240 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757528095.020582    1240 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757528095.048195    1240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757528095.048224    1240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757528095.048232    1240 computation_placer.cc:177] computation placer alr